```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```

# RMI Sample Queries: Data Engineer (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Engineer_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FData_Engineer_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Engineer_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Data_Engineer_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **Data Engineer** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `boston_oct_2025_sample_data` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### de1_materialized_view.sql
**Business Question**: Generate a query to create a 7-day materialized view of historical_travel_time for a specific corridor.
Product Stage: GA
Estimated Bytes Processed: ~150 MB
Metadata: Uses ALTER statements to apply technical descriptions to all columns and the view itself.

In [ ]:
%%bigquery --project {project_id} df_de1_materialized_view
-- NOTE: The source dataset (e.g., `boston_oct_2025_sample_data`) is a read-only subscription from Analytics Hub.
-- This materialized view MUST be created in a separate, writable dataset within your project.
-- Replace `your-project.your-dataset` with your target location.

CREATE MATERIALIZED VIEW IF NOT EXISTS `your-project.your-dataset.storrow_drive_view`
CLUSTER BY selected_route_id AS
SELECT
  selected_route_id,
  display_name,
  record_time,
  duration_in_seconds,
  static_duration_in_seconds,
  route_geometry
FROM `boston_oct_2025_sample_data.historical_travel_time`
WHERE record_time >= TIMESTAMP_SUB(TIMESTAMP('2025-10-31'), INTERVAL 7 DAY)
  AND display_name LIKE '%Storrow-Drive%';

-- Applying view-level metadata
ALTER MATERIALIZED VIEW `your-project.your-dataset.storrow_drive_view`
SET OPTIONS (
  description="A 7-day rolling subset of RMI historical travel time data specifically for the Storrow Drive corridor."
);

-- Applying column-level metadata descriptions
ALTER COLUMN selected_route_id SET OPTIONS(description="Unique identifier for the SelectedRoute resource.")
ON `your-project.your-dataset.storrow_drive_view`;

ALTER COLUMN display_name SET OPTIONS(description="User-provided descriptive name for the route.")
ON `your-project.your-dataset.storrow_drive_view`;

ALTER COLUMN record_time SET OPTIONS(description="The UTC timestamp representing when the route data was computed.")
ON `your-project.your-dataset.storrow_drive_view`;

ALTER COLUMN duration_in_seconds SET OPTIONS(description="The traffic-aware duration of the route in seconds.")
ON `your-project.your-dataset.storrow_drive_view`;

ALTER COLUMN static_duration_in_seconds SET OPTIONS(description="The traffic-unaware (static) duration of the route in seconds.")
ON `your-project.your-dataset.storrow_drive_view`;

ALTER COLUMN route_geometry SET OPTIONS(description="The traffic-aware polyline geometry of the route as a GEOGRAPHY object.")
ON `your-project.your-dataset.storrow_drive_view`;


### de2_data_cleaning.sql
**Business Question**: Write a query that produces a "cleaned" version of the routes_status table, correctly casting the route_length.
Product Stage: GA
Estimated Bytes Processed: < 1 MB
Metadata: Provides descriptions for transformed fields and the view itself.

In [ ]:
%%bigquery --project {project_id} df_de2_data_cleaning
/*
  PRE-REQUISITE: This query utilizes the custom routeAttribute 'route_length' 
  (intended physical length in meters), which has been pre-configured for 
  all routes in this sample dataset.
*/

CREATE OR REPLACE VIEW `your-project.your-dataset.routes_status_cleaned`
(
  selected_route_id OPTIONS(description="Unique identifier for the SelectedRoute resource."),
  display_name OPTIONS(description="User-provided descriptive name for the route."),
  status OPTIONS(description="Operational state (e.g., STATUS_RUNNING)."),
  validation_error OPTIONS(description="Reason for failure if status is INVALID."),
  route_length_meters OPTIONS(description="The pre-computed intended route length in meters, cast from the custom 'route_length' routeAttribute.")
)
OPTIONS(
  description="A cleaned view of SelectedRoutes status, with the custom route_length attribute promoted to a typed column."
)
AS
SELECT
  selected_route_id,
  display_name,
  status,
  validation_error,
  CAST(JSON_VALUE(route_attributes, '$.route_length') AS FLOAT64) AS route_length_meters
FROM `boston_oct_2025_sample_data.routes_status`
WHERE status != 'STATUS_INVALID';


### de3_sri_flattening.sql
**Business Question**: Create an optimized script to transform the latest 30 minutes of nested SRI data into a flattened format with spatial progress metrics and quality filters.
Product Stage: GA
Estimated Bytes Processed: ~10 MB (Optimized via Scripting and Static Partition Pruning)

In [ ]:
%%bigquery --project {project_id} df_de3_sri_flattening
/*
  BIGQUERY OPTIMIZATION PATTERN: Static vs. Dynamic Partition Pruning
  
  This query uses BigQuery Scripting (DECLARE/SET) to force "Static Pruning".
  
  1. Static Pruning (This Pattern): By resolving 'target_time' into a variable BEFORE 
     the main SELECT, BigQuery treats it as a constant. This allows the optimizer 
     to immediately discard irrelevant partitions.
     
  2. Geometry Integrity Check: To ensure high-quality analysis, this query:
     a) Calculates 'length_deviation_ratio' against pre-computed attributes.
     b) Excludes 'MultiLineString' geometries to ensure we only process single, 
        continuous paths (ST_LineString).
        
  3. Noise Reduction: Final results exclude 'NORMAL' speed states and filter out 
     extremely short intervals (< 5 meters) that often represent GPS noise.
*/

-- Step 1: Define the static anchor date to narrow down partitions
DECLARE anchor_date DATE DEFAULT '2025-10-31';

-- Step 2: Find the exact latest timestamp and define the 30-minute window
DECLARE latest_timestamp TIMESTAMP;
SET latest_timestamp = (
  SELECT MAX(record_time)
  FROM `boston_oct_2025_sample_data.recent_roads_data`
  WHERE record_time >= TIMESTAMP(anchor_date)
);

-- Step 3: Execute the flattening logic for the latest 30-minute window
WITH base_intervals AS (
  SELECT
    r.selected_route_id,
    r.record_time,
    segment_offset as interval_index,
    sri.speed as interval_speed_state,
    -- Reconstruct the interval polyline from the array of interval points
    ST_MAKELINE(sri.interval_coordinates) as interval_geometry,
    -- Core metrics for integrity check
    ST_LENGTH(r.route_geometry) as actual_route_length_meters,
    CAST(JSON_VALUE(s.route_attributes, '$.route_length') AS FLOAT64) as intended_route_length_meters
  FROM `boston_oct_2025_sample_data.recent_roads_data` r
  JOIN `boston_oct_2025_sample_data.routes_status` s USING(selected_route_id),
  UNNEST(speed_reading_intervals) AS sri WITH OFFSET AS segment_offset
  WHERE r.record_time >= TIMESTAMP(anchor_date)
    -- Capture only records from the last 30 minutes
    AND r.record_time > TIMESTAMP_SUB(latest_timestamp, INTERVAL 30 MINUTE)
    -- Quality filter: Only process single, continuous paths
    AND ST_GEOMETRYTYPE(r.route_geometry) = 'ST_LineString'
),
quality_filtered_intervals AS (
  SELECT
    *,
    -- Deviation between intended and actual geometry length
    SAFE_DIVIDE(ABS(actual_route_length_meters - intended_route_length_meters), intended_route_length_meters) as length_deviation_ratio
  FROM base_intervals
  -- Filter for high-integrity geometries (e.g., < 5% deviation)
  WHERE SAFE_DIVIDE(ABS(actual_route_length_meters - intended_route_length_meters), intended_route_length_meters) < 0.05
),
metrics_calculation AS (
  SELECT
    *,
    ST_LENGTH(interval_geometry) as interval_length_meters,
    -- Roll-up sum of interval lengths to find cumulative distance from origin
    SUM(ST_LENGTH(interval_geometry)) OVER (
      PARTITION BY selected_route_id, record_time 
      ORDER BY interval_index
    ) as cumulative_length_meters,
    -- Count total intervals in the route for context
    COUNT(*) OVER (
      PARTITION BY selected_route_id, record_time
    ) as total_intervals
  FROM quality_filtered_intervals
),
position_ratios AS (
  SELECT
    *,
    -- The end of the previous interval is the start of the current interval
    COALESCE(LAG(cumulative_length_meters) OVER (
      PARTITION BY selected_route_id, record_time 
      ORDER BY interval_index
    ), 0.0) as start_length_meters
  FROM metrics_calculation
)
SELECT
  selected_route_id,
  record_time,
  interval_index,
  total_intervals,
  interval_speed_state,
  interval_length_meters,
  -- Rounded relative positions (0.000 to 1.000) within the trip
  ROUND(SAFE_DIVIDE(start_length_meters, actual_route_length_meters), 3) as start_position_ratio,
  ROUND(SAFE_DIVIDE(cumulative_length_meters, actual_route_length_meters), 3) as end_position_ratio,
  length_deviation_ratio,
  interval_geometry
FROM position_ratios
-- Filter for congested intervals and exclude noise (short intervals)
WHERE interval_speed_state != 'NORMAL'
  AND interval_length_meters >= 5
ORDER BY selected_route_id, record_time, interval_index;


### de4_attribute_extraction.sql
**Business Question**: Write a query that pivots the JSON route_attributes into distinct columns.
Product Stage: GA
Estimated Bytes Processed: < 1 MB
Metadata: Enriches pivoted columns with business definitions.

In [ ]:
%%bigquery --project {project_id} df_de4_attribute_extraction
CREATE OR REPLACE VIEW `your-project.your-dataset.routes_enriched_attributes`
(
  selected_route_id OPTIONS(description="Unique identifier for the SelectedRoute resource."),
  region OPTIONS(description="The geographical business region extracted from routeAttributes."),
  tier OPTIONS(description="The service tier (e.g. priority, standard) extracted from routeAttributes."),
  priority OPTIONS(description="The operational priority level assigned during registration."),
  route_length_meters OPTIONS(description="The intended physical length of the route in meters, cast to FLOAT64 from routeAttributes.")
)
OPTIONS(
  description="A denormalized view of SelectedRoute metadata, promoting custom JSON attributes into typed top-level columns."
)
AS
SELECT
  selected_route_id,
  JSON_EXTRACT_SCALAR(route_attributes, '$.region') as region,
  JSON_EXTRACT_SCALAR(route_attributes, '$.tier') as tier,
  JSON_EXTRACT_SCALAR(route_attributes, '$.priority') as priority,
  -- route_attributes values are always strings. Casting to FLOAT64 for numerical analysis.
  CAST(JSON_EXTRACT_SCALAR(route_attributes, '$.route_length') AS FLOAT64) as route_length_meters
FROM `boston_oct_2025_sample_data.routes_status`
-- Example: Filtering by priority attribute
-- WHERE JSON_EXTRACT_SCALAR(route_attributes, '$.priority') = 'high';


### de5_freshness_audit.sql
**Business Question**: Which active routes have stopped receiving updates, indicating potential data gaps?
Product Stage: GA
Estimated Bytes Processed: ~151 MB
Metadata: Provides descriptions for the audit results.

In [ ]:
%%bigquery --project {project_id} df_de5_freshness_audit
/*
  AUDIT GOAL: Identify routes that are 'STATUS_RUNNING' but have no recent 
  records in historical_travel_time. This helps detect routes with 
  insufficient traffic or pipeline latency issues.
*/

CREATE OR REPLACE VIEW `your-project.your-dataset.route_freshness_audit`
(
  selected_route_id OPTIONS(description="Unique identifier for the SelectedRoute resource."),
  display_name OPTIONS(description="Human-readable name of the route."),
  last_updated OPTIONS(description="The timestamp of the most recent record found in historical_travel_time."),
  hours_since_last_update OPTIONS(description="The age of the data in hours relative to the audit timestamp.")
)
OPTIONS(
  description="Operational audit view to identify active routes with missing or stale travel time data."
)
AS
WITH freshness AS (
  SELECT
    selected_route_id,
    MAX(record_time) as last_updated
  FROM `boston_oct_2025_sample_data.historical_travel_time`
  -- Scans the full sample month to find the latest record for every route
  WHERE record_time BETWEEN '2025-10-01' AND '2025-11-01'
  GROUP BY 1
)
SELECT
  s.selected_route_id,
  s.display_name,
  f.last_updated,
  -- Using '2025-11-01' as the reference 'Now' for this static sample dataset
  TIMESTAMP_DIFF(TIMESTAMP('2025-11-01'), f.last_updated, HOUR) AS hours_since_last_update
FROM `boston_oct_2025_sample_data.routes_status` s
LEFT JOIN freshness f USING(selected_route_id)
-- Focus on routes that SHOULD be providing data
WHERE s.status = 'STATUS_RUNNING'
ORDER BY hours_since_last_update DESC;


### de7_routes_status_snapshot.sql
**Business Question**: How can I automate the historical tracking of my SelectedRoutes' status changes?
Product Stage: GA
Estimated Bytes Processed: < 1 MB
Metadata: Inherits column descriptions from routes_status and adds snapshot metadata.

In [ ]:
%%bigquery --project {project_id} df_de7_routes_status_snapshot
/*
  AUTOMATION EXAMPLE: 
  To schedule this snapshot daily at 2 AM UTC using the bq CLI, run:
  
  bq mk \
    --transfer_config \
    --project_id="your-project-id" \
    --data_source=scheduled_query \
    --display_name="Daily RMI Routes Status Snapshot" \
    --target_dataset="your_dataset" \
    --schedule="every 24 hours" \
    --params='{
      "query":"INSERT INTO `your-project.your-dataset.routes_status_history` SELECT CURRENT_TIMESTAMP() as snapshot_time, * FROM `boston_oct_2025_sample_data.routes_status`"
    }'
*/

-- STEP 1: Initialize the partitioned history table with enriched metadata
CREATE TABLE IF NOT EXISTS `your-project.your-dataset.routes_status_history` (
  snapshot_time TIMESTAMP OPTIONS(description="The UTC timestamp when this snapshot was captured."),
  selected_route_id STRING OPTIONS(description="Unique identifier for the SelectedRoute resource."),
  display_name STRING OPTIONS(description="User-provided descriptive name for the route."),
  status STRING OPTIONS(description="Current operational state (e.g., STATUS_RUNNING, STATUS_INVALID)."),
  validation_error STRING OPTIONS(description="Detailed reason if the route failed validation."),
  low_road_usage_start_time TIMESTAMP OPTIONS(description="Timestamp when low road usage was first detected."),
  route_attributes STRING OPTIONS(description="JSON string of custom business metadata.")
)
PARTITION BY DATE(snapshot_time)
CLUSTER BY selected_route_id;

-- STEP 2: The Periodic Append Logic (Manually executable version)
-- This statement appends the current state of all routes into the history table.
INSERT INTO `your-project.your-dataset.routes_status_history`
SELECT
  CURRENT_TIMESTAMP() as snapshot_time,
  selected_route_id,
  display_name,
  status,
  validation_error,
  low_road_usage_start_time,
  route_attributes
FROM `boston_oct_2025_sample_data.routes_status`;
